In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("HF_TOKEN")

In [2]:
import os
os.environ['HF_TOKEN'] = secret_value
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)

In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-24 06:42:52.665415: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779604973.102650      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779604973.218926      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779604974.449688      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779604974.449750      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779604974.449753      57 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.6: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.5.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [14]:
from datasets import load_from_disk

def filter_invalid(dataset):
    n_original = dataset.num_rows
    filtered = dataset.filter(lambda x: x['目标数量'] != -1, keep_in_memory=True, load_from_cache_file=False)

    n_invalid = n_original - filtered.num_rows
    print(f"Removed {n_invalid} invalid rows. {filtered.num_rows} rows remaining.")

    return filtered

dataset = load_from_disk("/kaggle/input/datasets/kakuate/tsa-train")
dataset = filter_invalid(dataset)

Filter:   0%|          | 0/6000 [00:00<?, ? examples/s]

Removed 25 invalid rows. 5975 rows remaining.


In [9]:
for example in dataset.select(range(5000, 5025)):
    print(example['标签'])

[-1, 1, 1, 1, 1]
[0, 0, 1, -1, 0, 0, -1]
[1, 1, 1, 1, 1, 0, 1, 1, 1, 1]
[1, 1, 1, 0, 1, -1, -1, 0, -1, 0]
[1, 1, 1, 1, 0, 1, 1]
[1, 1, 1, 1, 1, 1]
[0, -1, -1, -1, -1, -1, 1]
[1, 1, 1, 1, 1, 1]
[0, 1, -1, 1, 1]
[]
[0, 0, -1, -1, 1, 1, 0, 1]
[1, 1, 1, 1, 1]
[-1, -1]
[-1, -1, -1, -1, -1]
[0, -1, -1, -1]
[1, -1, 1, 0, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 1, 0, 1, 0]
[1, 1, 1, 1, 1, 0, 1, 1, 1]
[1, 1, 1]
[-1, 0, 0, -1]
[1, 1, 1, -1]
[1, 1, 0, 1, 1, 1, 0, 1, 1]
[1, 1, 0, 1, 1, -1]
[1, 1, 1, 1]


In [7]:
from datasets import load_from_disk

train_mapped = load_from_disk("/kaggle/input/datasets/kakuate/tsa-augmented")

In [8]:
import os
# 1. 强行把缓存塞进可写目录
os.environ["HF_HOME"] = "/tmp/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets_cache"

from unsloth.chat_templates import standardize_data_formats
train_mapped = train_mapped.map(lambda x: x, keep_in_memory=True)
train_mapped = standardize_data_formats(train_mapped)
train_mapped

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

Dataset({
    features: ['评论', '目标', '目标数量', '任务', '标签', '理由', 'conversations'],
    num_rows: 7000
})

In [9]:
train_mapped[1]

{'评论': '我非常气愤！环境这么好，评价也不错的餐厅竟然会让我和老公吃得上吐下泻，还发烧了！今天差一点回不来杭州！我和老公平时肠胃都很好！昨天晚上6吃了你家198双人套餐以后大概12点起来拉肚子，跟着难受呕吐！老公也吐了还发烧！一晚上来来回回好几次！我从来没有这样过！我估计是意大利面有问题吧应为我吃的少老公吃的多他比较严重！我们在你家就餐前五个小时是没有吃过任何东西的！太让人愤慨了！我现在回杭州打了针好一些了！要是我还在鼓浪屿我一定去你家说清楚！不能把不好的东西给消费者吃吧！你家又不是路边摊没有卫生许可证！黑心商家！希望大家看了我的点评千万不要到他家吃东西了！像我一样糟糕了！太让人失望了！当时我选择你家还是看了大众点评，感觉环境也不错！怎么会是这样的结果！',
 '目标': ['环境', '双人套餐', '意大利面'],
 '目标数量': 3,
 '任务': 'T3',
 '标签': [1, -1, -1],
 '理由': ['评论中提到“环境这么好”、“感觉环境也不错”，明确表达了对餐厅环境的满意与喜爱。',
  '用户和丈夫在食用198元双人套餐后，都出现了上吐下泻、发烧等严重食物中毒症状，用户强烈谴责该套餐导致身体不适，情感极为负面。',
  '用户根据自己吃得少、丈夫吃得多且症状更重的情况，合理推测意大利面是导致食物中毒的主要原因，情感为负面。'],
 'conversations': [{'role': 'system',
   'content': '\n你是一名精通目标情感分析（TSA）的数据科学家。在工作中，你必须严格遵守以下输出格式规范：\n\n【全局输出格式规范】\n你必须且只能输出一个标准的 JSON 对象，严禁包裹任何 Markdown 语法标记（如 ```json）。该对象必须严格包含中文键名，绝对不得出现任何英文键名。\n\n【行为边界】\n你只需输出符合上述规范的 JSON 数据，不得输出任何前导词、解释性文字或后续总结。确保 JSON 格式绝对合法。\n'},
  {'role': 'user',
   'content': '【任务指令】\n请提取后面评论中的所有目标实体，并给出相应的情感标签（1代表正向，0代表中性，-1代表负向）。\n\n【示例】\n评论：这新键盘敲击声真是一场“听觉盛宴”，办公室同事都快被我吵得掀桌子了；不

In [11]:
train = dataset.select(range(5000))

In [4]:
train

Dataset({
    features: ['评论', '交通方便', '位于商圈附近', '是否容易寻找', '排队时间', '服务人员态度', '停车方便', '点菜/上菜速度', '价格', '性价比', '折扣力度', '装修', '嘈杂情况', '就餐空间', '卫生情况', '分量', '口味', '外观', '食物推荐程度', '目标', '标签', '理由', '目标数量'],
    num_rows: 5000
})

In [5]:
train[0]

{'评论': '状元楼饭店第一次去，因为地理位置优越：在宁波市和义大道高、大、上，里面装修中式，菜是地道的宁波菜，口味纯正，醉泥螺特棒，吃到了小时候的味道，因为去了晚了，在大堂等了一会儿，期间有茶水喝、服务员还与你聊天，到了就餐时生意太好，服务员都是小跑状，服务态度绝对不提速，样样都服务到位，点酒水还耐心的与我们解释，就这样绝对要夸一夸，特别是彭新星、洪继华（看服务牌才知道名字）也给我们宁波市形象增色，状元楼是宁波的一扇窗口，服务员的素质更体现我们宁波人的精神面貌。赞一个',
 '交通方便': 1,
 '位于商圈附近': 1,
 '是否容易寻找': 1,
 '排队时间': -2,
 '服务人员态度': 1,
 '停车方便': -2,
 '点菜/上菜速度': -2,
 '价格': -2,
 '性价比': -2,
 '折扣力度': -2,
 '装修': 1,
 '嘈杂情况': -2,
 '就餐空间': -2,
 '卫生情况': -2,
 '分量': -2,
 '口味': 1,
 '外观': -2,
 '食物推荐程度': -2,
 '目标': ['地理位置', '装修', '菜品', '醉泥螺', '服务员'],
 '标签': [1, 1, 1, 1, 1],
 '理由': ['评论指出状元楼在宁波市和义大道，评价为“高、大、上”，地理位置优越，给顾客留下良好第一印象。',
  '评论描述装修为中式风格，属于正面评价，说明环境雅致。',
  '评论称赞菜是地道宁波菜，口味纯正，整体味道很好，包括醉泥螺等特色菜均有好评。',
  '醉泥螺被特别评价为“特棒”，并吃出了“小时候的味道”，蕴含怀旧情感，属于强烈正面。',
  '服务员在等位期间提供茶水并主动聊天，就餐时服务动作迅速、态度耐心、服务到位，尤其彭新星、洪继华两位服务员素质突出，被赞为宁波形象的窗口。'],
 '目标数量': 5}

In [9]:
ABLATION = "reasons"

if ABLATION == "reasons":
    print("w/o Reasons Mode.")
elif ABLATION == None:
    print("Full Mode.")
else:
    raise

w/o Reasons Mode.


In [12]:
def formatting_prompts_func(examples):
    # 初始化一个列表来存放所有处理好的对话
    conversations = []
    
    for i in range(len(examples["评论"])):
        # 构造符合 OpenAI/HuggingFace 标准的 messages 格式
        system_reason = "和理由"
        reply_reason = f", \"理由\": {examples['理由'][i]}"
        if ABLATION == "reasons":
            system_reason = ""
            reply_reason = ""
        reply_str = f"{{\"目标\": {examples['目标'][i]}, \"标签\": {examples['标签'][i]}" + reply_reason + f"}}"
        
        
        conversation = [
            {
                "role": "system", 
                "content": "你是一个评价分析专家，请从评论中提取评价目标、情感标签" + system_reason + "。"
            },
            {
                "role": "user", 
                "content": examples["评论"][i]
            },
            {
                "role": "assistant", 
                "content": reply_str
            }
        ]
        conversations.append(conversation)
    
    return {"conversations": conversations}

# 应用转换
train_mapped = train.map(formatting_prompts_func, batched=True, load_from_cache_file=False, cache_file_name = os.path.join(cache_dir, "train_cache.arrow"))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [13]:
train_mapped[0]

{'评论': '状元楼饭店第一次去，因为地理位置优越：在宁波市和义大道高、大、上，里面装修中式，菜是地道的宁波菜，口味纯正，醉泥螺特棒，吃到了小时候的味道，因为去了晚了，在大堂等了一会儿，期间有茶水喝、服务员还与你聊天，到了就餐时生意太好，服务员都是小跑状，服务态度绝对不提速，样样都服务到位，点酒水还耐心的与我们解释，就这样绝对要夸一夸，特别是彭新星、洪继华（看服务牌才知道名字）也给我们宁波市形象增色，状元楼是宁波的一扇窗口，服务员的素质更体现我们宁波人的精神面貌。赞一个',
 '交通方便': 1,
 '位于商圈附近': 1,
 '是否容易寻找': 1,
 '排队时间': -2,
 '服务人员态度': 1,
 '停车方便': -2,
 '点菜/上菜速度': -2,
 '价格': -2,
 '性价比': -2,
 '折扣力度': -2,
 '装修': 1,
 '嘈杂情况': -2,
 '就餐空间': -2,
 '卫生情况': -2,
 '分量': -2,
 '口味': 1,
 '外观': -2,
 '食物推荐程度': -2,
 '目标': ['地理位置', '装修', '菜品', '醉泥螺', '服务员'],
 '标签': [1, 1, 1, 1, 1],
 '理由': ['评论指出状元楼在宁波市和义大道，评价为“高、大、上”，地理位置优越，给顾客留下良好第一印象。',
  '评论描述装修为中式风格，属于正面评价，说明环境雅致。',
  '评论称赞菜是地道宁波菜，口味纯正，整体味道很好，包括醉泥螺等特色菜均有好评。',
  '醉泥螺被特别评价为“特棒”，并吃出了“小时候的味道”，蕴含怀旧情感，属于强烈正面。',
  '服务员在等位期间提供茶水并主动聊天，就餐时服务动作迅速、态度耐心、服务到位，尤其彭新星、洪继华两位服务员素质突出，被赞为宁波形象的窗口。'],
 '目标数量': 5,
 'conversations': [{'content': '你是一个评价分析专家，请从评论中提取评价目标、情感标签。',
   'role': 'system'},
  {'content': '状元楼饭店第一次去，因为地理位置优越：在宁波市和义大道高、大、上，里面装修中式，菜是地道的宁波菜，口味纯正，醉泥螺特棒，吃到了小时候的味道，因为去了晚了，在大堂等了一会儿，期间有茶水喝、服务员

In [12]:
def apply_template(examples):
    # 使用 tokenizer 的渲染功能
    # tokenize=False 表示只生成文本字符串，不转成 ID（通常微调器会自动 tokenize）
    # add_generation_prompt=False 因为我们已经提供了 assistant 的回答
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["conversations"]
    ]
    return {"text": texts}

# 这里的 remove_columns 顺便把你之前担心的多余列全删了
train_filtered = train_mapped.map(
    apply_template,
    batched=True,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
    cache_file_name="/kaggle/working/final_dataset.arrow"
)

NameError: name 'dataset' is not defined

In [13]:
train_filtered = train_mapped.map(
    apply_template,
    batched=True,
    remove_columns=['评论', '目标', '目标数量', '任务', '标签', '理由'],
    load_from_cache_file=False,
    cache_file_name="/kaggle/working/final_dataset.arrow"
)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [14]:
train_filtered

Dataset({
    features: ['conversations', 'text'],
    num_rows: 7000
})

In [15]:
train_filtered[1]

{'conversations': [{'role': 'system',
   'content': '\n你是一名精通目标情感分析（TSA）的数据科学家。在工作中，你必须严格遵守以下输出格式规范：\n\n【全局输出格式规范】\n你必须且只能输出一个标准的 JSON 对象，严禁包裹任何 Markdown 语法标记（如 ```json）。该对象必须严格包含中文键名，绝对不得出现任何英文键名。\n\n【行为边界】\n你只需输出符合上述规范的 JSON 数据，不得输出任何前导词、解释性文字或后续总结。确保 JSON 格式绝对合法。\n'},
  {'role': 'user',
   'content': '【任务指令】\n请提取后面评论中的所有目标实体，并给出相应的情感标签（1代表正向，0代表中性，-1代表负向）。\n\n【示例】\n评论：这新键盘敲击声真是一场“听觉盛宴”，办公室同事都快被我吵得掀桌子了；不过它的重量很沉，金属外壳做工倒是挺扎实的。\n\n输出JSON：\n{\n    "目标": ["噪音控制", "重量", "外观工艺"],\n    "标签": [-1, 0, 1],\n    "理由": [\n        "评论通过‘听觉盛宴’和‘同事快被吵得掀桌子’来反讽键盘吵闹，实际评价对象是‘噪音控制’，态度负向，故标签为-1。",\n        "评论客观陈述‘重量很沉’，属于不带感情色彩的事实描述，实际评价对象是‘重量’，态度中性，故标签为0。",\n        "评论直言‘外壳做工挺扎实’表达了对产品质量的认可，实际评价对象是‘外观工艺’，态度正向，故标签为1。"\n    ]\n}\n\n【待分析评论】\n评论：我非常气愤！环境这么好，评价也不错的餐厅竟然会让我和老公吃得上吐下泻，还发烧了！今天差一点回不来杭州！我和老公平时肠胃都很好！昨天晚上6吃了你家198双人套餐以后大概12点起来拉肚子，跟着难受呕吐！老公也吐了还发烧！一晚上来来回回好几次！我从来没有这样过！我估计是意大利面有问题吧应为我吃的少老公吃的多他比较严重！我们在你家就餐前五个小时是没有吃过任何东西的！太让人愤慨了！我现在回杭州打了针好一些了！要是我还在鼓浪屿我一定去你家说清楚！不能把不好的东西给消费者吃吧！你家又不是路边摊没有卫生许可证！黑心商家！希望大家看了我的点评

In [12]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_filtered,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        output_dir = "tsa-qwen-ckpts",
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 1,
        logging_steps = 10,

        num_train_epochs = 3, # Set this for 1 full training run.
        warmup_steps = 100,

        push_to_hub = True,
        hub_strategy = "checkpoint",
        hub_model_id = "TicklingShell/tsa-qwen-ckpts",
        hub_private_repo = True,
        
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 1018,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from trl import SFTTrainer, SFTConfig
from huggingface_hub import snapshot_download
from transformers import trainer_utils

local_dir = "./tsa-qwen-ckpts"
repo_id = "TicklingShell/tsa-qwen-ckpts"

resume_checkpoint = None

# 1. 检查本地有没有，如果没有（说明会话重置了），去 HF 远程下载最新的 Checkpoint
if not os.path.exists(local_dir) or not trainer_utils.get_last_checkpoint(local_dir):
    print("🌐 本地无记录，正在从 Hugging Face 远程下载最新进度...")
    try:
        # 只下载最新的 checkpoint 相关文件，不下载根目录的其他冗余权重
        snapshot_download(
            repo_id = repo_id,
            local_dir = local_dir,
            allow_patterns = "checkpoint-*/*" 
        )
        print("✅ 远程 Checkpoint 下载完成！")
    except Exception as e:
        print(f"🌱 远程仓库尚无数据或下载失败（原因: {e}），将从头开始训练。")

# 2. 自动定位最新的 checkpoint 文件夹
if os.path.exists(local_dir):
    last_checkpoint = trainer_utils.get_last_checkpoint(local_dir)
    if last_checkpoint is not None:
        print(f"🔥 完美衔接！将从断点续训: {last_checkpoint}")
        resume_checkpoint = last_checkpoint

# 3. 正常拉起训练
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_filtered,
    args = SFTConfig(
        output_dir = local_dir,
        save_strategy = "steps",
        save_steps = 5,
        save_total_limit = 2,
        logging_steps = 5,

        num_train_epochs = 3, # Set this for 1 full training run.
        warmup_steps = 50,

        push_to_hub = True,
        hub_strategy = "checkpoint",
        hub_model_id = repo_id,
        hub_private_repo = True,
        
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 1018,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train(resume_from_checkpoint = resume_checkpoint)

🌐 本地无记录，正在从 Hugging Face 远程下载最新进度...
✅ 远程 Checkpoint 下载完成！


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 3 | Total steps = 657
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
5,2.307100
10,2.245700
15,2.164700
20,1.992300
25,1.817100
30,1.583500
35,1.376800
40,1.181400
45,1.068800
50,1.032200


KeyboardInterrupt: 

In [17]:
from trl import SFTTrainer, SFTConfig
from huggingface_hub import snapshot_download
from transformers import trainer_utils

local_dir = "./tsa-qwen-ckpts"
repo_id = "TicklingShell/tsa-qwen-ckpts"

resume_checkpoint = None

snapshot_download(
            repo_id = repo_id,
            local_dir = local_dir,
            # allow_patterns = "checkpoint-*/*" 
        )

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/264M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/6.29k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

'/kaggle/working/tsa-qwen-ckpts'

In [18]:
type(trainer_utils.get_last_checkpoint(local_dir))

NoneType

In [15]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                     {"目标": [\'朝天锅\', \'地理位置\', \'环境\', \'服务\'], "标签": [1, -1, 0, 1], "理由": [\'点了全猪、五香肉和素菜，味道都很好，女朋友也觉得比想象的好吃，整体口味令人满意。\', \'作为外地人，店铺不太好找，需要依赖导航才能到达，说明位置有些隐蔽。\', \'小店环境普通，但干净整洁，可以接受，没有明显的好评或差评。\', \'店主一家态度特好，像一家人一样热情，服务体验温馨。\']}<|im_end|>\n'

In [15]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [25]:
if os.path.exists(local_dir):
    last_checkpoint = trainer_utils.get_last_checkpoint(local_dir)
    if last_checkpoint is not None:
        print(f"🔥 完美衔接！将从断点续训: {last_checkpoint}")
        resume_checkpoint = last_checkpoint

🔥 完美衔接！将从断点续训: ./tsa-qwen-ckpts/checkpoint-790


In [19]:
resume_checkpoint = os.path.join(local_dir, "last-checkpoint")
resume_checkpoint

'./tsa-qwen-ckpts/last-checkpoint'

In [42]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_dir,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.5.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [21]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_filtered,
    args = SFTConfig(
        output_dir = local_dir,
        save_strategy = "steps",
        save_steps = 5,
        save_total_limit = 2,
        logging_steps = 5,

        num_train_epochs = 3, # Set this for 1 full training run.
        warmup_steps = 50,

        push_to_hub = True,
        hub_strategy = "checkpoint",
        hub_model_id = repo_id,
        hub_private_repo = True,
        
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 1018,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train(resume_from_checkpoint = resume_checkpoint)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 3 | Total steps = 657
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
565,0.667700
570,0.636500
575,0.658100
580,0.655500
585,0.618400
590,0.654200
595,0.632900
600,0.650000
605,0.650600
610,0.647900


In [20]:
trainer_stats = trainer.train(resume_from_checkpoint = resume_checkpoint)

NameError: name 'trainer' is not defined

In [28]:
USER_PROMPT = """
你是一个精通针对性情感分析（TSA）的评价分析专家。
请仔细阅读用户提供的评论，从中精准提取：
1. 评价目标（目标对象）
2. 情感标签（1代表积极，-1代表消极，0代表中性）
3. 理由（支撑该情感的简短词句）

【约束条件】
- 必须输出标准的 JSON 对象，所有的键和字符串值必须使用双引号（"）。
- 不要包含任何 Markdown 格式标记（如 ```json ），不要有任何前言或后语。
- 三个列表的长度必须完全一致，按照提取顺序一一对应。

【示例】
输入评论："这手机屏幕显示很细腻，但是电池掉电太快了。"
输出JSON：{"目标": ["屏幕", "电池"], "标签": [1, -1], "理由": ["显示很细腻", "掉电太快了"]}

【提供评论】
输入评论：
"""

In [8]:
SYSTEM_PROMPT = """
你是一名精通目标情感分析（TSA）的数据科学家。你的核心任务是分析用户评论，提取被评价的目标实体，并评估其情感倾向。在工作中，你必须严格遵守以下输出格式规范：

【全局输出格式规范】
你必须且只能输出一个标准的 JSON 对象，严禁包裹任何 Markdown 语法标记（如 ```json）。该对象必须严格包含以下四个中文键名，绝对不得出现任何英文键名：
- "目标": 列表（List），由你提取出的标准实体名称字符串组成。
- "标签": 列表（List），与"目标"列表中的实体一一对应，填充情感倾向数字（1代表正向，0代表中性，-1代表负向）。
- "理由": 列表（List），与"目标"列表中的实体一一对应，详细阐述该目标获得此情感标签的具体依据。

【行为边界】
你只需输出符合上述规范的 JSON 数据，不得输出任何前导词、解释性文字或后续总结。确保 JSON 格式绝对合法。
"""

In [9]:
USER_PROMPT = """
【任务指令】
请对后面的评论执行深度“目标情感分析”（TSA）任务。请严格按照以下步骤进行分析：
1. 实体提取与边界划定：扫描全文，精准提取出所有被评价的目标实体（注意划定文本边界，并挖掘出未明说但实际被评价的“隐含实体”）。
2. 文本修辞与语境辨析：仔细分辨评论中对各实体的评价是否存在反讽、隐喻、夸张等情况，还原用户的真实表达意图。
3. 情感倾向处理：在排除修辞干扰后，判断最终真实情感倾向（1代表正向，0代表中性，-1代表负向）。

【输出格式约束】
你必须输出一个标准的JSON对象，且必须严格包含以下中文键名，不得出现任何英文键名：
- "目标": 列表，由你提取出的标准实体名称组成。
- "标签": 列表，与"目标"列表一一对应的情感倾向数字（1, 0, 或 -1）。
- "理由": 列表，与"目标"列表一一对应，详细阐述该目标获得此标签的具体依据。

【示例】
评论：这手机拍照真“清晰”，大白天拍人能拍出鬼影来，不过续航确实顶，用了一天还有一半电。

输出JSON：
{
    "思维链": "1. 实体提取：显式实体有‘续航’（用了一天还有电）。‘拍照真清晰’中提取出隐含实体‘拍照’。 2. 语境辨析：‘拍照真清晰’加了双引号，且后文提到‘拍出鬼影’，判定属于强烈的反讽修辞，真实意图是极度不满；‘续航确实顶’为夸张赞美，无反讽。 3. 情感处理：‘拍照’排除反讽干扰后为负向（-1），‘续航’为正向（1）。",
    "目标": ["拍照", "续航"],
    "标签": [-1, 1],
    "理由": [
        "评论使用反讽手法，表面夸清晰实际指出大白天拍出鬼影，对拍照功能极度不满。",
        "评论直言续航确实顶，并用具体数据（用了一天还有一半电）证实了对续航的强烈认可。"
    ]
}

【待分析评论】
评论：
"""

In [10]:
def tsa(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT + f"{example['评论']}\n输出JSON："}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True, # Must add for generation
    )
    
    output = model.generate(
        **tokenizer(text, return_tensors = "pt").to("cuda"),
        max_new_tokens = 1000, # Increase for longer outputs!
        temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    )
    
    output_ids = output[0][len(tokenizer(text, return_tensors = "pt").to("cuda").input_ids[0]):].tolist() 
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [11]:
def calculate_f1(y_true, y_pred):
    """
    y_true: 列表，每个元素是 set()，如 [{"A", "B"}, {"C"}]
    y_pred: 列表，每个元素是 set()，如 [{"A", "D"}, {"C"}]
    """
    # 初始化用于 Micro-F1 的全局计数器
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    # 用于存储每条样本 F1 的列表，用于计算 Macro-F1
    sample_f1_list = []

    for gold, pred in zip(y_true, y_pred):
        # 集合运算
        tp = len(gold & pred)       # 预测对的
        fp = len(pred - gold)       # 多预测的（幻觉）
        fn = len(gold - pred)       # 没预测到的（漏掉）
        
        # --- 计算 Micro 所需的全局累加 ---
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        # --- 计算当前样本的 F1 (用于 Macro) ---
        p = tp / len(pred) if len(pred) > 0 else 0
        r = tp / len(gold) if len(gold) > 0 else 0
        s_f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
        sample_f1_list.append(s_f1)

    # 1. 计算 Micro-F1
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = (2 * micro_p * micro_r) / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    
    # 2. 计算 Macro-F1 (样本平均)
    macro_f1 = sum(sample_f1_list) / len(sample_f1_list) if len(sample_f1_list) > 0 else 0
    
    return {
        "Micro-F1": round(micro_f1, 4),
        "Macro-F1": round(macro_f1, 4),
        "Micro-Precision": round(micro_p, 4),
        "Micro-Recall": round(micro_r, 4)
    }

In [12]:
import json
from tqdm import tqdm

val_split = dataset.select(range(5000, 5100))

# 分别存储仅目标，以及目标+情感的集合
golds_target = []
preds_target = []

golds_combined = []
preds_combined = []

for example in tqdm(val_split, "Eval"):
    output = tsa(example)
    
    # --- 处理 Gold (真值) ---
    target_set = set(example['目标'])
    label_set = set(zip(example['目标'], example['标签']))
    
    golds_target.append(target_set)
    golds_combined.append(label_set)
    
    # --- 处理 Prediction (预测值) ---
    try:
        json_body = json.loads(output)
        p_target = set(json_body['目标'])
        # 即使模型输出的列表长度不一，zip 也会按最短的处理，或者报错
        p_combined = set(zip(json_body['目标'], json_body['标签']))
        
        preds_target.append(p_target)
        preds_combined.append(p_combined)
    except:
        # 解析失败时，两组预测都存入空集
        preds_target.append(set())
        preds_combined.append(set())

# 计算仅实体的提取效果
print("Target Extraction F1:")
print(calculate_f1(golds_target, preds_target))

# 计算实体+情感都对齐的效果
print("Combined TSA F1:")
print(calculate_f1(golds_combined, preds_combined))

Eval:  61%|██████    | 61/100 [45:36<29:09, 44.86s/it]  


KeyboardInterrupt: 

In [13]:
# 计算仅实体的提取效果
print("Target Extraction F1:")
print(calculate_f1(golds_target, preds_target))

# 计算实体+情感都对齐的效果
print("Combined TSA F1:")
print(calculate_f1(golds_combined, preds_combined))

Target Extraction F1:
{'Micro-F1': 0.3825, 'Macro-F1': 0.3484, 'Micro-Precision': 0.4221, 'Micro-Recall': 0.3498}
Combined TSA F1:
{'Micro-F1': 0.3286, 'Macro-F1': 0.2983, 'Micro-Precision': 0.3626, 'Micro-Recall': 0.3005}


In [23]:
print(golds_combined[0], preds_combined[0])

{('服务', 1), ('羊血', 1), ('用餐环境', -1), ('价格', 1), ('红汤羊肉汤锅', 1)} set()


In [15]:
def calculate_conditional_f1(y_true_combined, y_pred_combined):
    """
    y_true_combined: list of set([(target, sentiment), ...])
    y_pred_combined: list of set([(target, sentiment), ...])
    """
    cond_true = []
    cond_pred = []

    for gold_set, pred_set in zip(y_true_combined, y_pred_combined):
        # 找出 gold 中的所有 target
        gold_targets = {t for t, s in gold_set}
        # 找出 pred 中的所有 target
        pred_targets = {t for t, s in pred_set}
        
        # 只有当 target 在两边都出现时，才把它们的情感标签放进对比池
        common_targets = gold_targets & pred_targets
        
        for t in common_targets:
            # 找到 gold 中该 target 对应的情感
            g_s = [s for target, s in gold_set if target == t][0]
            # 找到 pred 中该 target 对应的情感
            p_s = [s for target, s in pred_set if target == t][0]
            
            # 为了复用之前的 calculate_f1，我们包装成集合形式
            cond_true.append({g_s})
            cond_pred.append({p_s})

    # 调用你现有的 calculate_f1 函数
    return calculate_f1(cond_true, cond_pred)

In [16]:
calculate_conditional_f1(golds_combined, preds_combined)

{'Micro-F1': 0.8591,
 'Macro-F1': 0.8591,
 'Micro-Precision': 0.8591,
 'Micro-Recall': 0.8591}

In [34]:
dataset.select(range(5001, 5002))[0]['评论']

'前几天来南京出差的，建邺区的万达广场总部培训，中午和分所的小伙伴凑凑一起来吃的午饭，由于餐标实在太低了，吃个绿茶也勉勉强强，点了〈越南菌菇卷〉就是把金针菇包成一团的凉菜〈石锅有机花菜〉味道普普通通，还可以〈绿茶烤鸡〉要了半只，味道不错，挺嫩的〈绿茶烤肉〉也是招牌菜〈椒麻鸡〉量很大，但是是大葱的量惊人，鸡很少〈手捏菜炒蘑菇〉就是青菜梗炒蘑菇〈自家滚豆腐〉一般般〈糖醋里脊〉味道还不错，量太少了〈绿茶鱼排〉味道怪怪的〈烤肉酱油炒饭〉给男同胞点的主食，没尝'

In [41]:
tsa(dataset.select(range(5001, 5002))[0])

'{"目标": ["餐标", "越南菌菇卷", "石锅有机花菜", "绿茶烤鸡", "绿茶烤肉", "椒麻鸡", "手捏菜炒蘑菇", "自家滚豆腐", "糖醋里脊", "绿茶鱼排", "烤肉酱油炒饭"], "标签": [-1, 1, 0, 1, 1, 0, 0, 0, 0, -1, 0], "理由": ["餐标太低，吃个绿茶也勉勉强强", "越南菌菇卷味道不错", "味道普普通通，还可以", "味道不错，挺嫩的", "味道不错，挺嫩的", "量很大但大葱太多鸡太少", "青菜梗炒蘑菇，没有明确评价", "一般般", "味道还不错，但量太少了", "味道怪怪的", "没有尝，未给出明确评价"]}'

In [36]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")

('qwen_lora/tokenizer_config.json',
 'qwen_lora/special_tokens_map.json',
 'qwen_lora/chat_template.jinja',
 'qwen_lora/vocab.json',
 'qwen_lora/merges.txt',
 'qwen_lora/added_tokens.json',
 'qwen_lora/tokenizer.json')

In [22]:
model_id = "TicklingShell/tsa-qwen-lora-augmented-3eps"

model.push_to_hub(model_id, private=True) # Online saving
tokenizer.push_to_hub(model_id, private=True) # Online saving

README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/TicklingShell/tsa-qwen-lora-augmented-3eps


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
from unsloth import FastLanguageModel
import torch

model_id = "TicklingShell/tsa_qwen_lora" # 例如 "huggingface/llama-3-8b-lora"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = True, # 如果显存紧张，继续用4bit
)

==((====))==  Unsloth 2026.5.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [8]:
import json

def parse_model_output(raw_output, aspect_list):
    try:
        # 提取 JSON 部分
        data = json.loads(raw_output[raw_output.find('{'):raw_output.rfind('}')+1])
        # 按标准顺序转换，如果 Key 缺失则默认补 -2
        return [data.get(aspect, -2) for aspect in aspect_list]
    except:
        # 如果 JSON 解析失败，返回全为 -2 的列表（或报错）
        return [-2] * len(aspect_list)

# 你的 18 个方面顺序列表
aspect_list = [
    'Location#Transportation', 'Location#Downtown', 'Location#Easy_to_find', 
    'Service#Queue', 'Service#Hospitality', 'Service#Parking', 
    'Service#Timely', 'Price#Level', 'Price#Cost_effective', 
    'Price#Discount', 'Ambience#Decoration', 'Ambience#Noise', 
    'Ambience#Space', 'Ambience#Sanitary', 'Food#Portion', 
    'Food#Taste', 'Food#Appearance', 'Food#Recommend'
]

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = str(sample['review'])
system = """
### Role
你是一个精通 ABSA（基于方面的情感分析）的专家。你的任务是根据用户评论，严格按照 18 个预定义方面输出情感极性。

### Output Format (CRITICAL)
你必须输出一个 JSON 对象，Key 是给定的 18 个方面，Value 是整数。
- 未涉及：-2
- 正向：1
- 负向：-1
- 中性：0

不要输出任何解释文字，只输出合法的 JSON 字符串。

### Aspects List
1. Location#Transportation
2. Location#Downtown
3. Location#Easy_to_find
4. Service#Queue
5. Service#Hospitality
6. Service#Parking
7. Service#Timely
8. Price#Level
9. Price#Cost_effective
10. Price#Discount
11. Ambience#Decoration
12. Ambience#Noise
13. Ambience#Space
14. Ambience#Sanitary
15. Food#Portion
16. Food#Taste
17. Food#Appearance
18. Food#Recommend

### Example
Input: "菜很好吃，但是要排队。"
Output:
{
  "Location#Transportation": -2, "Location#Downtown": -2, "Location#Easy_to_find": -2,
  "Service#Queue": 0, "Service#Hospitality": -2, "Service#Parking": -2, "Service#Timely": -2,
  "Price#Level": -2, "Price#Cost_effective": -2, "Price#Discount": -2,
  "Ambience#Decoration": -2, "Ambience#Noise": -2, "Ambience#Space": -2, "Ambience#Sanitary": -2,
  "Food#Portion": -2, "Food#Taste": 1, "Food#Appearance": -2, "Food#Recommend": -2
}
"""
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

content: {
  "Location#Transportation": 1, "Location#Downtown": 1, "Location#Easy_to_find": 1,
  "Service#Queue": -1, "Service#Hospitality": 1, "Service#Parking": -2, "Service#Timely": -1,
  "Price#Level": -2, "Price#Cost_effective": -2, "Price#Discount": -2,
  "Ambience#Decoration": 1, "Ambience#Noise": -2, "Ambience#Space": -2, "Ambience#Sanitary": -2,
  "Food#Portion": -2, "Food#Taste": 1, "Food#Appearance": -2, "Food#Recommend": 1
}


In [12]:
pred_list = parse_model_output(content, aspect_list)
pred_list

[1, 1, 1, -1, 1, -2, -1, -2, -2, -2, 1, -2, -2, -2, -2, 1, -2, 1]

In [11]:
def dict_to_list(data_dict, order):
    # 使用 .get(key, -2) 增加容错性，如果某个键缺失则填充 -2
    return [data_dict.get(aspect, -2) for aspect in order]

# 4. 执行转换
true_list = dict_to_list(sample, aspect_list)
true_list

[1, 1, 1, -2, 1, -2, -2, -2, -2, -2, 1, -2, -2, -2, -2, 1, -2, -2]

In [14]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

def evaluate_detection(y_true, y_pred):
    # 将二维列表展平
    flat_true = np.array(y_true).flatten()
    flat_pred = np.array(y_pred).flatten()
    
    # 转换为二分类：1 代表有提到，0 代表未提及(-2)
    binary_true = (flat_true != -2).astype(int)
    binary_pred = (flat_pred != -2).astype(int)
    
    print("=== 提取能力评估 (Aspect Detection) ===")
    print(classification_report(binary_true, binary_pred, target_names=['未提及', '已提及']))
    return f1_score(binary_true, binary_pred)

# 调用
extract_f1 = evaluate_detection(true_list, pred_list)

=== 提取能力评估 (Aspect Detection) ===
              precision    recall  f1-score   support

         未提及       1.00      0.75      0.86        12
         已提及       0.67      1.00      0.80         6

    accuracy                           0.83        18
   macro avg       0.83      0.88      0.83        18
weighted avg       0.89      0.83      0.84        18



In [18]:
def evaluate_sentiment(y_true, y_pred):
    flat_true = np.array(y_true).flatten()
    flat_pred = np.array(y_pred).flatten()
    
    mask = flat_true != -2
    valid_true = flat_true[mask]
    valid_pred = flat_pred[mask]
    
    # 统计模型“完全漏掉”的情况（真实有，模型给-2）
    missed_count = np.sum(valid_pred == -2)
    total_aspects = len(valid_true)
    
    print(f"--- 情感分类细分分析 ---")
    print(f"真实存在的 Aspect 总数: {total_aspects}")
    print(f"模型完全没识别出来的（漏报）: {missed_count} ({missed_count/total_aspects:.2%})")
    
    # 计算 F1 时，将 -2 视为一种特殊的错误类别，或者直接看 labels=[ -1, 0, 1] 的表现
    # 注意：如果模型输出了 -2，而 labels 里没写 -2，那么这个样本在计算 Precision 时不计入
    # 但在计算 Recall 时会作为分母，导致 Recall 下降。
    report = classification_report(valid_true, valid_pred, 
                                   labels=[-1, 0, 1], 
                                   target_names=['Neg(-1)', 'Neu(0)', 'Pos(1)'],
                                   zero_division=0)
    print(report)

    macro_f1 = f1_score(valid_true, valid_pred, average='macro')
    return macro_f1

# 调用
sentiment_f1 = evaluate_sentiment(true_list, pred_list)
sentiment_f1

--- 情感分类细分分析 ---
真实存在的 Aspect 总数: 6
模型完全没识别出来的（漏报）: 0 (0.00%)
              precision    recall  f1-score   support

     Neg(-1)       0.00      0.00      0.00         0
      Neu(0)       0.00      0.00      0.00         0
      Pos(1)       1.00      1.00      1.00         6

    accuracy                           1.00         6
   macro avg       0.33      0.33      0.33         6
weighted avg       1.00      1.00      1.00         6



1.0

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = str(sample)
system = """
你是一个情感分析专家。你的任务是基于用户输入的评论数据和参考数据，分析其情感极性。

### 用户输入格式
Python字典：
'id'为数字id，无具体含义。
'review'对应具体评论内容。
'star'对应评价星级。
其余键名对应“具体方面的评价”，具体解释如下。

### 你的输出格式
输出JSON格式。模板如下：
{
    "极性": 1,
    "理由": <一段具体的分析>
}
其中，极性可能为1, -1, 0，分别对应正向，负向，中性。

### 注意事项
- “具体方面的评价”属于你的参考数据，你的理由分析必须是纯文字的。
- 当你看到参考数据为 -2 时，请在思考时表述为‘用户未提到某某方面’；
- 看到 1 时，表述为‘用户觉得某某方面很好’。
- 严禁在理由中直接抄写输入数据里的英文单词。

### “具体方面的评价”相关数值解释：
- "1"代表情感极性为正。
- "-1"代表情感极性为负。
- "0"代表情感极性为中性。
- "-2"代表原始评论并未涉及这一方面。请注意，虽然-2是负数，但此时其并非代表负向情感，而是没有任何情感倾向。

### 示例
输入：{'review': '菜很好吃', 'service': -2}
输出：{
    "极性": 1,
    "理由": "用户对菜品味道表示赞赏。虽然评价中未提及服务方面的内容，但整体用餐体验非常正面。"
}
解释：因为service的值为-2，所以评价中并没有提及服务方面的内容。
"""
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69fc8db5-2aac810350a8acbe01172a3c;97dae609-525c-41e3-b58b-f17d9cb621b7)

Entry Not Found for url: https://huggingface.co/api/models/Qwen/Qwen3-4B-Instruct-2507/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"